# Scalar transport verification for the new `micromet.py`

**Goal.** Check that the scalar profiles produced by the current
`pyAPES.microclimate.micromet` are correct, by running the *same* eddy diffusivity
through two scalar solvers:

- `closure_1_model_scalar_old` — the main-branch implementation, copied verbatim into
  this notebook (`tridiag`, arithmetic face diffusivities, no source term at node 0).
- `closure_1_model_scalar` — imported directly from `micromet.py` (`solve_banded`,
  harmonic mean at the lowest face, source term at node 0).

Momentum comes from `closure_1_model_U_fvm` (also imported from `micromet.py`) in
**both** cases, so any difference between the two profiles is a property of the scalar
solver alone.

Three plant area density profiles are used: `weibull` (a forest), `constant`
(grass / low shrub up to ~0.5-1 m) and `zero` (bare ground / snow).

**The yardstick** is a steady-state mass balance: the flux through the topmost interior
face must equal the ground flux plus everything released below it. That is a property of
the true solution, so any solver that fails it is wrong regardless of how the profile
looks.

In [ ]:
%matplotlib widget
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

# make the repository importable when the notebook is opened from debug_notebooks/
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyAPES').is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pyAPES.microclimate.micromet import closure_1_model_U_fvm, closure_1_model_scalar
from pyAPES.utils.utilities import tridiag, spatial_average
from pyAPES.utils.constants import EPS, VON_KARMAN, MOLAR_MASS_AIR, SPECIFIC_HEAT_AIR

sns.set_context('notebook')
print('repository root:', REPO_ROOT)

## The old scalar solver

Copied from the main branch (`pyAPES/microclimate/micromet.py`) with the name changed.
Mathematically identical to `closure_1_model_scalar_nodal` in the earlier diagnosis
notebook, but it keeps the production signature so the two solvers can be called with
exactly the same keyword arguments.

In [ ]:
def closure_1_model_scalar_old(dz: float, Ks: np.ndarray, source: np.ndarray,
                               ubc: float, lbc: float, scalar: str,
                               T: float = 20.0, P: float = 101300.0,
                               lbc_dirchlet: bool = False) -> np.ndarray:
    """
    Steady-state scalar profile, main-branch implementation.

    Kept verbatim apart from the name: Thomas algorithm, arithmetic face
    diffusivities throughout, and no source term in the budget of node 0.

    Args:
        dz (float): [m], grid increment
        Ks (array): [m2 s-1], eddy diffusivity at the nodes
        source (array): sink/source, CO2 [umol m-3 s-1], H2O [mol m-3 s-1], T [W m-3]
        ubc (float): value at the top node, CO2 [ppm], H2O [mol mol-1], T [degC]
        lbc (float): ground flux, CO2 [umol m-2 s-1], H2O [mol m-2 s-1], T [W m-2]
        scalar (str): 'CO2' | 'H2O' | 'T'
        T (float): [degC], air temperature for the molar density of air
        P (float): [Pa], ambient pressure
        lbc_dirchlet (bool): True for a fixed value at the ground
    Returns:
        (array): scalar profile, CO2 [ppm] | H2O [mol mol-1] | T [degC]
    """
    dz = float(dz)
    N = len(Ks)
    rho_a = P / (287.05 * (T + 273.15))          # [kg m-3], air density
    CF = rho_a / MOLAR_MASS_AIR                  # [mol m-3], molar conc. of air

    Ks = spatial_average(Ks, method='arithmetic')  # length N+1, face values

    if scalar.upper() == 'CO2':                  # [umol] -> [mol]
        ubc = 1e-6 * ubc
        source = 1e-6 * source
        lbc = 1e-6 * lbc

    if scalar.upper() == 'T':                    # [J m-3 K-1], volumetric heat capacity
        CF = CF * SPECIFIC_HEAT_AIR

    a, b, g, f = (np.zeros(N) for _ in range(4))

    a[1:-1] = Ks[1:-2]
    b[1:-1] = -(Ks[1:-2] + Ks[2:-1])
    g[1:-1] = Ks[2:-1]
    f[1:-1] = -source[1:-1] / CF * dz ** 2

    a[-1], b[-1], g[-1], f[-1] = 0.0, 1.0, 0.0, ubc          # Dirichlet at the top

    if not lbc_dirchlet:                                      # flux at the ground
        a[0], b[0], g[0], f[0] = 0.0, 1.0, -1.0, (lbc / CF) * dz / (Ks[1] + EPS)
    else:                                                     # fixed value
        a[0], b[0], g[0], f[0] = 0.0, 1.0, 0.0, lbc

    x = tridiag(a, b, g, f)

    if scalar.upper() == 'CO2':                  # [mol] -> [umol]
        x = 1e6 * x

    return x

## Scenario

The grid, forcing and fluxes are the ones pyAPES normally runs with: `z = 0 ... 25 m`,
`dz = 0.25 m`. The canopy fluxes are distributed vertically in proportion to the plant
area density; the ground fluxes are the lower boundary condition of the scalar solvers.

In [ ]:
# --- grid and surface
z_top = 25.0          # [m], top of the domain / measurement height
dz = 0.25             # [m], model grid increment
z0 = 0.01             # [m], forest floor roughness length (micromet['zos'])
z = np.arange(0.0, z_top + 0.5 * dz, dz)

# --- forcing
U_ref = 5.0           # [m s-1], wind speed at z_top
dPdx = 0.0            # [-], u*-normalized horizontal pressure gradient
Cd = 0.2              # [-], drag coefficient

# --- canopy shapes
LAI_weibull = 4.0     # [m2 m-2]
hc_weibull = 15.0     # [m], canopy height
hb_weibull = 3.0      # [m], crown base height
LAI_const = 1.0       # [m2 m-2], grass / low shrub
h_const = 0.5         # [m], top of the uniform layer

# --- scalars, pyAPES convention Ks = Sc * Km
Sc = {'CO2': 2.0, 'H2O': 2.0, 'T': 2.0}
T_top = 15.0          # [degC]
CO2_top = 400.0       # [ppm]
H2O_top = 0.012       # [mol mol-1]
P_air = 101300.0      # [Pa]

# canopy fluxes, distributed in proportion to LAD
F_co2_canopy = -15.0  # [umol m-2 s-1], net assimilation (negative = uptake)
F_h2o_canopy = 3.0    # [mmol m-2 s-1], transpiration
F_h_canopy = 150.0    # [W m-2], sensible heat

# ground fluxes, the lower boundary condition
F_co2_ground = 3.0    # [umol m-2 s-1], soil respiration
F_h2o_ground = 0.5    # [mmol m-2 s-1], ground evaporation
F_h_ground = 20.0     # [W m-2], ground sensible heat

# u* scale from the neutral log law, and the normalized upper boundary condition
u_star = VON_KARMAN * U_ref / np.log(z_top / z0)
Utop_n = U_ref / u_star

rho_air = P_air / (287.05 * (T_top + 273.15))    # [kg m-3]
CF = rho_air / MOLAR_MASS_AIR                    # [mol m-3], molar conc. of air

print(f'grid   : {len(z)} nodes, {z[0]:.2f} ... {z[-1]:.2f} m, dz = {dz} m')
print(f'u*     : {u_star:.4f} m s-1, Utop/u* = {Utop_n:.3f}')
print(f'CF     : {CF:.2f} mol m-3')
print(f'canopy resolved by mixing_length_fvm only if hc > 3*dz = {3 * dz} m')

In [ ]:
def make_lad(z: np.ndarray, kind: str, LAI: float, hc: float = 0.0, hb: float = 0.0,
             h_const: float = 1.0, b: float = 0.906, c: float = 2.145) -> tuple:
    """
    Plant area density profile, normalised so that sum(LAD*dz) = LAI.

    The Weibull shape follows lad_weibul in pyAPES.utils.utilities (Teske and
    Thistle 2004, parameters of Scots pine by default).

    Args:
        z (array): [m], grid, constant increment
        kind (str): 'zero' | 'constant' | 'weibull'
        LAI (float): [m2 m-2], leaf area index, the scaling factor
        hc (float): [m], canopy height, used by 'weibull'
        hb (float): [m], crown base height, used by 'weibull'
        h_const (float): [m], top of the uniform layer, used by 'constant'
        b, c (float): Weibull shape parameters
    Returns:
        (tuple):
            lad (array): [m2 m-3], one-sided plant area density
            h_eff (float): [m], canopy height seen by the solvers
    """
    z = np.asarray(z, dtype=float)
    dz = z[1] - z[0]
    a = np.zeros(len(z))

    if kind == 'zero':
        return a, 0.0

    if kind == 'constant':
        ix = np.where((z > 0.0) & (z <= h_const))[0]
        a[ix] = 1.0
        h_eff = h_const

    elif kind == 'weibull':
        ix = np.where((z > hb) & (z <= hc))[0]
        x = np.linspace(0.0, 1.0, len(ix))       # normalized within-crown height
        a[ix] = np.abs(-(c / b) * (((1.0 - x) / b) ** (c - 1.0))
                       * np.exp(-((1.0 - x) / b) ** c)
                       / (1.0 - np.exp(-(1.0 / b) ** c)))
        h_eff = hc

    else:
        raise ValueError("kind must be 'zero', 'constant' or 'weibull'")

    a = a / (np.sum(a) * dz)                     # integral of the shape function = 1
    return LAI * a, h_eff

In [ ]:
# the three canopies; CASES keeps (lad, hc) so every later cell loops over the same dict
CASES = {}
CASES['weibull'] = make_lad(z, 'weibull', LAI_weibull, hc=hc_weibull, hb=hb_weibull)
CASES['constant'] = make_lad(z, 'constant', LAI_const, h_const=h_const)
CASES['zero'] = make_lad(z, 'zero', 0.0)

STYLE = {'weibull': ('tab:green', '-', f'weibull, LAI = {LAI_weibull}, hc = {hc_weibull} m'),
         'constant': ('tab:orange', '--', f'constant, LAI = {LAI_const}, hc = {h_const} m'),
         'zero': ('tab:blue', '-.', 'zero (bare ground / snow)')}

for name, (lad, hc) in CASES.items():
    print(f'{name:9}: integral(LAD dz) = {np.sum(lad) * dz:.4f} m2 m-2, hc = {hc:.2f} m, '
          f'{"resolved" if hc >= 3 * dz else "UNRESOLVED -> open-ground branch"}')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 6))

for name, (lad, hc) in CASES.items():
    color, ls, label = STYLE[name]
    ax.plot(lad, z, ls, color=color, label=label)

ax.set_xlabel('LAD [m$^2$ m$^{-3}$]')
ax.set_ylabel('z [m]')
ax.set_ylim(0, z_top)
ax.legend(frameon=False, fontsize=9)
ax.text(0.02, 0.98, 'a)', transform=ax.transAxes, va='top')
fig.tight_layout()

## Momentum

`closure_1_model_U_fvm` is called once per canopy. It returns everything on the input
grid (the cell faces of its own finite-volume discretisation, which are exactly the
pyAPES nodes), so `U`, `Km` and `tau` line up with `z` and with the source terms without
any interpolation.

`tau` comes back normalised by the flux through the topmost interior face, so
`ustar[-1] = u*` by construction and `ustar[0]` is the true surface friction velocity.

In [ ]:
def solve_momentum(z: np.ndarray, lad: np.ndarray, hc: float) -> dict:
    """
    Wind, shear stress and eddy diffusivity from micromet.closure_1_model_U_fvm,
    scaled to dimensional units with the u* of the scenario.

    Args:
        z (array): [m], node grid, constant increment
        lad (array): [m2 m-3], one-sided plant area density
        hc (float): [m], canopy height
    Returns:
        (dict): dimensional U [m s-1], Km [m2 s-1], ustar [m s-1], normalized tau,
            mixing length, displacement height, roughness length and the surface
            conductance g_m
    """
    tau_n, U_n, Km_n, l_mix, d, zo, gm = closure_1_model_U_fvm(
        z, Cd, lad, hc, Utop_n, z0=z0, dPdx=dPdx)

    return {'U': U_n * u_star,                       # [m s-1]
            'Km': Km_n * u_star,                     # [m2 s-1]
            'tau': tau_n,                            # [-], normalized by tau_top
            'ustar': np.sqrt(np.abs(tau_n)) * u_star,  # [m s-1]
            'l_mix': l_mix, 'd': d, 'zo': zo, 'gm': gm}

In [ ]:
flow = {name: solve_momentum(z, lad, hc) for name, (lad, hc) in CASES.items()}

for name, f in flow.items():
    print(f"{name:9}: d = {f['d']:6.3f} m, zo = {f['zo']:.4e} m, g_m = {f['gm']:.5f}, "
          f"U(0.25 m) = {f['U'][1]:.4f} m/s, ustar(surface) = {f['ustar'][0]:.4f} m/s")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
ax = axes.ravel()

for name, f in flow.items():
    color, ls, label = STYLE[name]
    ax[0].plot(f['U'], z, ls, color=color, label=label)
    ax[1].plot(f['tau'], z, ls, color=color, label=label)
    ax[2].plot(f['Km'], z, ls, color=color, label=label)

ax[0].set_xlabel('U [m s$^{-1}$]')
ax[1].set_xlabel(r'$\tau/\tau_{top}$ [-]')
ax[2].set_xlabel('$K_m$ [m$^2$ s$^{-1}$]')
for a, letter in zip(ax, 'abc'):
    a.set_ylabel('z [m]')
    a.set_yscale('log')
    a.set_ylim(0.5 * dz, z_top)
    a.text(0.02, 0.98, f'{letter})', transform=a.transAxes, va='top')
ax[0].legend(frameon=False, fontsize=9)
fig.suptitle('Momentum from micromet.closure_1_model_U_fvm')
fig.tight_layout()

## A third momentum solver, and the log law as ground truth

`closure_1_model_U_fvm_opa` below is copied verbatim from
`debug_notebooks/u_profile_and_scalar_transport_diagnosis.ipynb` (only the helper
`mixing_length_opa` is renamed from its notebook copy to avoid shadowing
`mixing_length_fvm`, which `micromet.py` already exports under its own name). It solves
the same equations as `closure_1_model_U_fvm` — same conductance boundary condition,
same drag term, same mixing length formula — but keeps the Dirichlet upper boundary
condition fixed at the top **cell centre** instead of correcting it for the half-cell
offset to `z_faces[-1]`.

With `LAD = 0` there is an exact reference to check both solvers against: the neutral
log law, $U(z) = (u_*/\kappa)\,\ln(z/z_0)$. Any deviation from it is a property of the
discretisation, not of the physics, since the log law is the exact solution of the
governing ODE on open ground.

In [ ]:
def mixing_length_opa(z: np.ndarray, z0: float, hc: float, d: float,
                      dz: float = None, l_min: float = None) -> np.ndarray:
    """
    Turbulent mixing length, as used by closure_1_model_U_fvm_opa.

    Identical in every respect to micromet.mixing_length_fvm; kept as a separate copy
    only so this notebook can call the two solvers with their own, unmodified helper
    functions side by side.
    """
    dz = z[1] - z[0] if dz is None else dz

    if l_min is None:
        l_min = VON_KARMAN * z0

    l_ground = VON_KARMAN * z

    if hc < 3 * dz:   # canopy not resolved by the grid -> open-ground branch
        return np.maximum(l_ground, l_min)

    alpha = (hc - d) * VON_KARMAN / (hc + EPS)
    I_F = np.sign(z - hc) + 1.0
    l_mix = alpha * hc * (1 - I_F / 2) + (I_F / 2) * (VON_KARMAN * (z - d))

    sc = (alpha * hc) / VON_KARMAN
    ix = np.where(z < sc)
    l_mix[ix] = l_ground[ix]

    return np.maximum(l_mix, l_min)


def closure_1_model_U_fvm_opa(z: np.ndarray, z0: float, Cd: float, lad: np.ndarray,
                              hc: float, Utop: float, Ubot: float = 0.0,
                              dPdx: float = 0.0, lbc: str = 'conductance', gamma: float = 0.5,
                              l_min: float = None, max_iter: int = 200):
    """
    Copied verbatim from debug_notebooks/u_profile_and_scalar_transport_diagnosis.ipynb,
    cell 'closure_1_model_U_fvm_opa'. Not modified: this is the version being compared
    against, not the target implementation.
    """
    z_faces = z.copy()  # z input is faces
    z_midpoint = 0.5 * (z_faces[:-1] + z_faces[1:]) # z_midpoint has shape (N-1,).

    lad_faces = 0.5 * np.asarray(lad, dtype=float)
    lad = 0.5 * (lad_faces[:-1] + lad_faces[1:])
    dz = z_midpoint[1] - z_midpoint[0]

    N = len(z_midpoint)

    if l_min is None:
        l_min = VON_KARMAN * z0

    if lbc == 'conductance':
        g_tau = (VON_KARMAN / np.log(z_midpoint[0] / z0)) ** 2
    else:
        raise NotImplementedError("only lbc='conductance' is implemented")

    # Initial guess for U profile
    U = np.linspace(max(Ubot, EPS), Utop, N)
    err = 1e6
    iter_no = 0

    # Define RHS of the matrix equation outside iteration loop as this never changes
    rhs = np.full(N, -1.0*dPdx) # This needs to be minus if we have vertical pressure gradient. In OPT notes we only deal with system where dP/dx=0
    rhs[-1] = Utop
    while err > 1e-6 and iter_no < max_iter:
        iter_no += 1

        Fd = Cd * lad * U ** 2 # drag force from last iteration
        d = np.sum(z_midpoint * Fd) / (np.sum(Fd) + EPS) # displacement height as centroid of drag force
        l_mix = mixing_length_opa(z_faces[1:-1], z0, hc, d, l_min=l_min) # mixing length at cell faces (not midpoints since flux is at faces)

        dUdz = (U[1:] - U[:-1]) / dz # calculate dUdz at elements which have element upwards and downwards from them
        #dUdz[0] = (U[0]) / dz0 # calculate dUdz at the bottom flux. Here we assume U(z0) = 0
        #dUdz[-1] = (Utop - U[-1]) / dz # calculate dUdz at the top flux
        Km = l_mix ** 2 * np.abs(dUdz)

        #Km_mean = 2 * Km[:-1] * Km[1:] / (Km[:-1] + Km[1:] + EPS) # harmonic mean of Km_i and Km_(i +/- 1), shape (N-1,)

        A_plus = Km[1:] / dz ** 2 # Check OPT notes from 25.8.2026 for naming of these matrix elements, shape (N-2,)
        A_minus = Km[:-1] / dz ** 2 # shape (N-2,)
        B = -A_plus - A_minus - Cd*lad[1:-1]*U[1:-1] # shape (N-2)
        C = -Km[0]/dz**2 - g_tau/dz*U[0]-Cd*lad[0]*U[0] # shape (1,)

        # Build tridiagonal matrix for solve_banded
        ab = np.zeros((3, N))
        ab[0, 2:] = A_plus  # upper diagonal
        ab[0, 1] = Km[0]/dz**2 # upper element for the conductance BC
        ab[1, 1:-1] = B # main diagonal
        ab[1, 0] = C # conductance BC at bottom
        ab[1,-1] = 1 # Dirichlet BC at top
        ab[2, :-2] = A_minus # lower diagonal

        U_new = solve_banded((1, 1), ab, rhs)
        err = np.max(np.abs(U_new - U))
        U = gamma * U_new + (1.0 - gamma) * U

    l_mix = mixing_length_opa(z_faces[1:-1], z0, hc, d, l_min=l_min)
    dUdz = (U[1:] - U[:-1]) / dz
    Km_int = l_mix ** 2 * np.abs(dUdz) #Km at interior faces

    # Create face arrays at original z which is what we want to return
    tau_f = np.zeros(N+1)
    Km_f = np.zeros_like(tau_f)
    U_f = np.zeros_like(tau_f)

    tau_f[1:-1]= Km_int * dUdz
    Km_f[1:-1] = Km_int
    U_f[1:-1] = 0.5*(U[1:] + U[:-1]) # mean between two adjacent cell centers

    #ground face: the conductance carries the whole unresolved layer from z0 to z[0]
    #Km_f[0] is the diffusivity that reprocudes tau_0 in this layer
    #since Km_f[0] (U[0]-0)/dz0 = g_tau U[0]**2 <=> Km_f[0] = g_tau U[0] dz0
    dz0 = z_midpoint[0] - z0
    tau_f[0] = g_tau*U[0]**2
    Km_f[0] = g_tau*U[0]*dz0
    U_f[0] = 0.0 #U(z0) = 0, here we approzimate that U(z=0) = U(z0) = 0

    # top face: Dirichlet BC
    tau_f[-1] = tau_f[-2]
    Km_f[-1] = Km_f[-2]
    U_f[-1] = U[-1] # Dirichlet BC at top

    return {'z': z_midpoint, 'z_f': z_faces, 'U': U, 'U_f': U_f,
            'tau': tau_f,
            'tau_f': tau_f, 'tau_surface': tau_f[0],
            'Km_f': Km_f, 'l_mix_f': l_mix,
            'l_mix': mixing_length_opa(z_midpoint, z0, hc, d, l_min=l_min),
            'd': d, 'gm': g_tau, 'grid': 'cell', 'l_form': 'max',
            'iterations': iter_no, 'err': err, 'converged': err <= 1e-6}

In [ ]:
def solve_momentum_opa(z: np.ndarray, lad: np.ndarray, hc: float) -> dict:
    """
    Wind, shear stress and eddy diffusivity from closure_1_model_U_fvm_opa, scaled to
    dimensional units and reshaped to the same keys as solve_momentum so the two can be
    plotted and tabulated together. tau_f is normalized by the flux through the
    topmost interior face here, in this wrapper, exactly as closure_1_model_U_fvm does
    internally -- fvm_opa itself returns the raw, unnormalized flux.

    Args:
        z (array): [m], node grid, constant increment
        lad (array): [m2 m-3], one-sided plant area density
        hc (float): [m], canopy height
    Returns:
        (dict): same keys as solve_momentum's return value
    """
    res = closure_1_model_U_fvm_opa(z, z0, Cd, lad, hc, Utop_n, dPdx=dPdx)
    tau_n = res['tau_f'] / (res['tau_f'][-2] + EPS)

    return {'U': res['U_f'] * u_star,
            'Km': res['Km_f'] * u_star,
            'tau': tau_n,
            'ustar': np.sqrt(np.abs(tau_n)) * u_star,
            'l_mix': res['l_mix_f'], 'd': res['d'], 'gm': res['gm'],
            'iterations': res['iterations'], 'converged': res['converged']}


flow_opa = {name: solve_momentum_opa(z, lad, hc) for name, (lad, hc) in CASES.items()}

for name in CASES:
    f, g = flow[name], flow_opa[name]
    print(f"{name:9}: fvm     d = {f['d']:6.3f} m, U(0.25 m) = {f['U'][1]:.4f} m/s")
    print(f"{'':9}  fvm_opa d = {g['d']:6.3f} m, U(0.25 m) = {g['U'][1]:.4f} m/s"
          f"  ({g['iterations']} iterations, converged = {g['converged']})")

### The log law, and why `closure_1_model_U_fvm` misses it

With `lad = 'zero'` the momentum equation reduces to a constant-flux layer, whose exact
solution is the log law. `mixing_length_fvm` evaluates the mixing length
$\ell = \kappa z$ at the **arithmetic** face height (`z_faces[1:-1]`, i.e. the actual grid
faces 0.25, 0.50, ... m). That is the right height for a linear profile, but $U(z)$ is not
linear — it is $\propto \ln z$, which is concave, and the finite-difference gradient
$(U_{i+1}-U_i)/\Delta z$ is the exact derivative not at the face itself but at the
**log-mean** height $\Delta z / \ln(z_{i+1}/z_i)$, which sits below the arithmetic face
height whenever the profile curves. Evaluating $\ell$ at the wrong height under- or
over-estimates $K_m$ there, and the error is largest exactly where the curvature of the
log profile is largest: close to the ground, where $z$ is only a few multiples of $z_0$.

A second, distinct error stacks on top of it: `closure_1_model_U_fvm` returns `U` on the
node grid by averaging the two neighbouring cell-centre values,
`U_f[1:-1] = 0.5*(U[1:] + U[:-1])`. Because $\ln$ is concave, the arithmetic mean of two
points on the curve is always **below** the curve's value at their midpoint, so this
reconstruction step subtracts a second, independent bias from the momentum error already
present in the cell-centre solution.

Both errors shrink with height (the log profile's curvature $\propto 1/z^2$ flattens out)
and both are inherent to the discretisation, not a coding bug: this is the same
arithmetic-vs-log-mean face height trade-off that was deliberately kept in
`closure_1_model_U_fvm_opa` because a log-mean face height only helps the open-ground
case and does not generalise to canopies. It affects both solvers identically at the
lowest few faces, since both call the mixing length at the same, uncorrected face
locations.

In [ ]:
z_log = np.logspace(np.log10(z0), np.log10(z_top), 300)
U_log = u_star / VON_KARMAN * np.log(z_log / z0)                # dimensional log law
U_log_at_z = u_star / VON_KARMAN * np.log(np.maximum(z, z0) / z0)  # sampled on the node grid

rows = []
for i in range(1, len(z)):
    rows.append({'z [m]': z[i],
                 'fvm error [%]': 100 * (flow['zero']['U'][i] / U_log_at_z[i] - 1),
                 'fvm_opa error [%]': 100 * (flow_opa['zero']['U'][i] / U_log_at_z[i] - 1)})
log_law_error_table = pd.DataFrame(rows).set_index('z [m]')
print('Error against the log law, LAD = 0, worst and best of the profile:')
print(log_law_error_table.iloc[[0, 1, 2, 3, -1]].round(4).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
ax = axes.ravel()

for name, f in flow.items():
    color, ls, label = STYLE[name]
    ax[0].plot(f['U'], z, ls, color=color, label=f'{label} (fvm)')
    ax[1].plot(f['U'], z, ls, color=color, label=f'{label} (fvm)')
    ax[2].plot(f['tau'], z, ls, color=color, label=label)
    ax[3].plot(f['Km'], z, ls, color=color, label=label)

for name, f in flow_opa.items():
    color, _, label = STYLE[name]
    ax[0].plot(f['U'], z, ':', color=color, label=f'{label} (fvm_opa)')
    ax[1].plot(f['U'], z, ':', color=color, label=f'{label} (fvm_opa)')

for a in (ax[0], ax[1]):
    a.plot(U_log, z_log, '-', color='0.2', lw=2, label='log law (exact, LAD = 0)')

ax[0].set_xlabel('U [m s$^{-1}$]')
ax[1].set_xlabel('U [m s$^{-1}$]')
ax[2].set_xlabel(r'$\tau/\tau_{top}$ [-]')
ax[3].set_xlabel('$K_m$ [m$^2$ s$^{-1}$]')

ax[1].set_yscale('log')
ax[1].set_ylim(0.5 * z0, 2.0)          # zoom on the surface layer: the diagnostic panel
for a, letter in zip(ax, 'abcd'):
    a.set_ylabel('z [m]')
    if letter != 'b':
        a.set_ylim(0.5 * dz, z_top)
    if letter in ('c', 'd'):
       a.set_yscale('log')
    a.text(0.02, 0.98, f'{letter})', transform=a.transAxes, va='top')
ax[0].legend(frameon=False, fontsize=8, loc='upper left')
fig.suptitle('Momentum: fvm vs. fvm_opa vs. the analytic log law (panel b is the surface-layer zoom)')
fig.tight_layout()

## Source terms

The canopy fluxes are spread over the layers in proportion to LAD, exactly as
`CanopyModel` does with `sources[key] += ... / self.dz`. With `LAD = 0` the whole column
is source-free and the only forcing is the ground flux, which makes that case an exact
constant-flux test.

In [ ]:
def canopy_sources(z: np.ndarray, lad: np.ndarray) -> dict:
    """
    Distribute the prescribed canopy fluxes vertically in proportion to LAD.

    Args:
        z (array): [m], node grid
        lad (array): [m2 m-3], plant area density
    Returns:
        (dict): 'CO2' [umol m-3 s-1], 'H2O' [mol m-3 s-1], 'T' [W m-3],
            i.e. the units closure_1_model_scalar expects for each scalar
    """
    dz_local = z[1] - z[0]
    total = np.sum(lad) * dz_local
    shape = lad / total if total > 0 else np.zeros(len(z))   # [m-1]
    return {'CO2': F_co2_canopy * shape,
            'H2O': 1e-3 * F_h2o_canopy * shape,
            'T': F_h_canopy * shape}


# boundary conditions per scalar, in the units closure_1_model_scalar expects
SCALARS = {'CO2': dict(ubc=CO2_top, lbc=F_co2_ground, unit='ppm'),
           'H2O': dict(ubc=H2O_top, lbc=1e-3 * F_h2o_ground, unit='mol mol$^{-1}$'),
           'T': dict(ubc=T_top, lbc=F_h_ground, unit='$^\\circ$C')}

## Scalar profiles, old vs. new

Both solvers get the identical `Ks = Sc * Km` from the same momentum solution and the
identical sources and boundary conditions. The only thing that differs is the solver.

In [ ]:
profiles = {}
for name, (lad, hc) in CASES.items():
    src = canopy_sources(z, lad)
    entry = {'source': src}
    for key, bc in SCALARS.items():
        Ks = np.maximum(Sc[key] * flow[name]['Km'], 1e-6)     # as Micromet.scalar_profiles
        kwargs = dict(dz=dz, Ks=Ks, source=src[key], ubc=bc['ubc'], lbc=bc['lbc'],
                      scalar=key, T=T_top, P=P_air)
        entry[key] = {'Ks': Ks,
                      'old': closure_1_model_scalar_old(**kwargs),
                      'new': closure_1_model_scalar(**kwargs)}
    profiles[name] = entry

for name in CASES:
    print(f'--- {name}')
    for key in SCALARS:
        p = profiles[name][key]
        print(f"  {key:4} node 0: old = {p['old'][0]:12.5f}, new = {p['new'][0]:12.5f}   "
              f"node 1: old = {p['old'][1]:12.5f}, new = {p['new'][1]:12.5f}")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 12), sharey=True)

for row, name in enumerate(CASES):
    for col, key in enumerate(SCALARS):
        a = axes[row, col]
        p = profiles[name][key]
        a.plot(p['old'], z, '-', color='tab:red', label='old (main branch)')
        a.plot(p['new'], z, '--', color='tab:blue', label='new (micromet.py)')
        a.set_yscale('log')
        a.set_ylim(0.5 * dz, z_top)
        a.set_xlabel(f"{key} [{SCALARS[key]['unit']}]")
        if col == 0:
            a.set_ylabel(f'{name}\nz [m]')

for a, letter in zip(axes.ravel(), 'abcdefghi'):
    a.text(0.02, 0.98, f'{letter})', transform=a.transAxes, va='top')
axes[0, 0].legend(frameon=False, fontsize=9)
fig.suptitle('Scalar profiles, same $K_s$ from closure_1_model_U_fvm')
fig.tight_layout()

## The test: steady-state mass balance

Node $j$ owns the cell $[z_j - \Delta z/2,\ z_j + \Delta z/2]$, except node 0 whose
half-cell is $[0,\ \Delta z/2]$; the topmost node is Dirichlet and has no budget. In
steady state the flux through the topmost interior face must equal the ground flux plus
everything released below it:

$$F_{top} = F_{ground} + \sum_j S_j h_j, \qquad h_0 = \tfrac{\Delta z}{2},\ h_{j>0} = \Delta z$$

This holds for the exact solution, so a solver that misses it is wrong however plausible
its profile looks.

In [ ]:
def scalar_balance(C: np.ndarray, Ks: np.ndarray, dz: float, source: np.ndarray,
                   lbc: float, scalar: str, T: float = 20.0,
                   P: float = 101300.0) -> tuple:
    """
    Steady-state mass balance of a scalar profile on the node grid.

    Args:
        C (array): converged profile, in the units the solver returned
        Ks (array): [m2 s-1], eddy diffusivity at the nodes (before face averaging)
        dz (float): [m], grid increment
        source (array): source on the same grid, same units as the solver input
        lbc (float): ground flux, same units as the solver input
        scalar (str): 'CO2' | 'H2O' | 'T'
        T (float): [degC], air temperature
        P (float): [Pa], ambient pressure
    Returns:
        (tuple): flux through the topmost interior face, the flux the budget
            requires, and their relative difference
    """
    rho_a = P / (287.05 * (T + 273.15))
    CF_local = rho_a / MOLAR_MASS_AIR
    if scalar.upper() == 'T':
        CF_local = CF_local * SPECIFIC_HEAT_AIR

    Ksf = spatial_average(Ks, method='arithmetic')
    F_top = -CF_local * Ksf[-2] * (C[-1] - C[-2]) / dz

    h = np.full(len(C), dz)
    h[0] = 0.5 * dz                                   # half-cell at the ground
    F_expected = lbc + np.sum(source[:-1] * h[:-1])   # top node has no budget

    return F_top, F_expected, (F_top - F_expected) / (abs(F_expected) + EPS)

In [ ]:
rows = []
for name in CASES:
    for key, bc in SCALARS.items():
        p = profiles[name][key]
        src = profiles[name]['source'][key]
        bo = scalar_balance(p['old'], p['Ks'], dz, src, bc['lbc'], key, T_top, P_air)
        bn = scalar_balance(p['new'], p['Ks'], dz, src, bc['lbc'], key, T_top, P_air)
        rows.append({'case': name, 'scalar': key,
                     'F required': bo[1],
                     'F_top old': bo[0], 'error old [%]': 100 * bo[2],
                     'F_top new': bn[0], 'error new [%]': 100 * bn[2]})

balance_table = pd.DataFrame(rows).set_index(['case', 'scalar'])
balance_table.round(6)

## Isolating where the difference comes from

Two things changed in `closure_1_model_scalar` besides the switch to `solve_banded`:
the harmonic mean at the lowest face, and the source term in the half-cell of node 0.
The reference implementation below is the new solver with the lower-boundary right-hand
side written as the half-cell budget requires,

$$-\frac{F_{ground}\,\Delta z}{CF} - \frac{S_0\,\Delta z^2}{2\,CF}$$

and with the harmonic mean switchable, so the two changes can be separated from each
other and from anything else.

In [ ]:
def closure_1_model_scalar_reference(dz: float, Ks: np.ndarray, source: np.ndarray,
                                     ubc: float, lbc: float, scalar: str,
                                     T: float = 20.0, P: float = 101300.0,
                                     harmonic_face: bool = True) -> np.ndarray:
    """
    Reference for the new solver: same banded assembly, lower boundary written
    straight from the half-cell budget of node 0.

    Args:
        as closure_1_model_scalar, plus
        harmonic_face (bool): True takes the harmonic mean at the lowest face,
            False keeps the arithmetic mean of the old solver
    Returns:
        (array): scalar profile
    """
    dz = float(dz)
    N = len(Ks)
    rho_a = P / (287.05 * (T + 273.15))
    CF_local = rho_a / MOLAR_MASS_AIR

    Ksf = spatial_average(Ks, method='arithmetic')
    if harmonic_face:
        # the lowest face spans the unresolved layer: resistances in series
        Ksf[1] = 2.0 * Ksf[0] * Ksf[1] / (Ksf[0] + Ksf[1] + EPS)

    if scalar.upper() == 'CO2':
        ubc = 1e-6 * ubc
        source = 1e-6 * source
        lbc = 1e-6 * lbc
    if scalar.upper() == 'T':
        CF_local = CF_local * SPECIFIC_HEAT_AIR

    ab = np.zeros((3, N))
    rhs = np.zeros(N)

    ab[2, 0:N - 2] = Ksf[1:-2]
    ab[1, 1:-1] = -(Ksf[1:-2] + Ksf[2:-1])
    ab[0, 2:] = Ksf[2:-1]
    rhs[1:-1] = -source[1:-1] / CF_local * dz ** 2

    ab[1, -1] = 1.0                                   # Dirichlet at the top
    rhs[-1] = ubc

    ab[1, 0] = -Ksf[1]                                # half-cell budget at the ground
    ab[0, 1] = Ksf[1]
    rhs[0] = -(lbc * dz / CF_local) - source[0] * dz ** 2 / (2.0 * CF_local)

    x = solve_banded((1, 1), ab, rhs)

    if scalar.upper() == 'CO2':
        x = 1e6 * x

    return x

In [ ]:
rows = []
for name in CASES:
    src = profiles[name]['source']
    for key, bc in SCALARS.items():
        p = profiles[name][key]
        kwargs = dict(dz=dz, Ks=p['Ks'], source=src[key], ubc=bc['ubc'], lbc=bc['lbc'],
                      scalar=key, T=T_top, P=P_air)
        ref_arith = closure_1_model_scalar_reference(**kwargs, harmonic_face=False)
        ref_harm = closure_1_model_scalar_reference(**kwargs, harmonic_face=True)
        b = scalar_balance(ref_harm, p['Ks'], dz, src[key], bc['lbc'], key, T_top, P_air)
        rows.append({
            'case': name, 'scalar': key,
            'reference balance error [%]': 100 * b[2],
            'max |old - reference(arith)|': np.max(np.abs(p['old'] - ref_arith)),
            'surface: old': p['old'][0],
            'surface: reference(harm)': ref_harm[0],
            'surface: micromet.py': p['new'][0]})

reference_table = pd.DataFrame(rows).set_index(['case', 'scalar'])
reference_table

### How to read the two tables

- **`max |old - reference(arith)|` at round-off** means the `tridiag` → `solve_banded`
  conversion is exact: with the arithmetic face mean the banded assembly reproduces the
  old solver node for node.
- **`reference balance error [%]` at round-off** means the half-cell lower boundary
  conserves mass, so the reference is the correct target.
- The remaining difference between `reference(harm)` and `old` sits **only at node 0**
  and is the intended one: the harmonic mean puts the resistance of the unresolved layer
  between $z_0$ and the first face into the surface value.
- Any non-zero `error new [%]` in the balance table is a defect of the current
  `micromet.py`, not of the discretisation.

## Short canopies: where `mixing_length_fvm` gives up

`mixing_length_fvm` takes the open-ground branch when `hc < 3*dz`. Just above that
threshold it takes the canopy branch instead, and for a short, dense canopy the drag
centroid `d` can land **above** `hc`. Then `alpha = (hc - d)*kappa/hc` is negative, the
in-canopy mixing length is clamped to `l_min` and the wind collapses. The scan below
shows where that happens on this grid.

In [ ]:
rows = []
for h in (0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0):
    lad_h, hc_h = make_lad(z, 'constant', LAI_const, h_const=h)
    f = solve_momentum(z, lad_h, hc_h)
    alpha = (hc_h - f['d']) * VON_KARMAN / (hc_h + EPS)
    rows.append({'h_const [m]': h,
                 'branch': 'open ground' if hc_h < 3 * dz else 'canopy',
                 'd [m]': f['d'],
                 'alpha': alpha,
                 'U(0.25 m) [m/s]': f['U'][1],
                 'Km(0.25 m) [m2/s]': f['Km'][1]})

short_canopy_table = pd.DataFrame(rows).set_index('h_const [m]')
short_canopy_table

## Conclusions

**1. The `tridiag` → `solve_banded` conversion is exact.** With the arithmetic face mean
kept, the banded assembly reproduces the old solver node for node
(`max |old - reference(arith)|` is at round-off, 1e-11 or smaller for every case and
every scalar). The band layout in `micromet.py` is right.

**2. The harmonic mean at the lowest face does what it should.** It differs from the old
solver **only at node 0** and raises the surface value, because it adds the resistance of
the unresolved layer between $z_0$ and the first face — the layer whose diffusivity
`closure_1_model_U_fvm` reports as $K_{m,0} = g_m U_0 (z_{c,0} - z_0)$. That is the whole
point of the surface-conductance boundary condition.

**3. The lower boundary right-hand side in `micromet.py` carries one $\Delta z$ too many.**
The line

```python
rhs[0] = -(lbc*dz / CF)*dz - source[0] * dz**2 / (2*CF)
```

applies `lbc*dz**2/CF` where the half-cell budget of node 0 needs `lbc*dz/CF`. The ground
flux that actually enters the column is therefore $\Delta z$ times too small — with
`dz = 0.25` exactly a quarter of it. The `zero` rows of the balance table show it
unambiguously: with no canopy sources the only forcing is the ground flux, and the new
solver delivers `error new = -75.000 %` for CO2, H2O and T alike. In the canopy cases the
deficit is the same absolute amount, $0.75\,F_{ground}$, diluted by the canopy sources
(-18.75 % for CO2, -10.71 % for H2O, -8.82 % for T).

The fix is to drop the second `dz`:

```python
rhs[0] = -(lbc * dz / CF) - source[0] * dz**2 / (2*CF)
```

`closure_1_model_scalar_reference` in this notebook is that line and nothing else changed;
its balance error is at round-off for all nine combinations.

**4. One thing to check separately: which pair the harmonic mean is taken of.**
`micromet.py` runs `spatial_average` first and then overwrites `Ks[1]` using `Ks[0]` and
`Ks[1]` of the **face** array, i.e. it combines $K_{s,0}^{node}$ with
$\tfrac12(K_{s,0}^{node} + K_{s,1}^{node})$. Taking the harmonic mean of the two **node**
values instead is the more literal reading of "two resistances in series". The difference
is small but it is a deliberate choice, so it is worth writing down which one is intended.

**5. `mixing_length_fvm` fails for short, dense canopies.** The scan in the previous cell
shows the mechanism: for a uniform canopy taller than the `hc < 3*dz` threshold the drag
centroid `d` lands **above** `hc`, `alpha = (hc - d)*kappa/hc` goes negative, the in-canopy
mixing length is clamped to `l_min = kappa*z0` and the wind collapses by four orders of
magnitude. On this grid every `h_const` between 0.75 m and 2 m is affected. This is why the
`constant` case in this notebook uses `h_const = 0.5 m`, which falls into the open-ground
branch and behaves sensibly. It is a momentum-solver issue, not a scalar one, but grass
and low shrub canopies land exactly in the broken range.

**6. `closure_1_model_U_fvm` does not reproduce the log law exactly, by design of the
discretisation, not by a bug.** With `LAD = 0` the error against the analytic log law is
largest at the lowest face (`z = 0.25 m`: `-4.59 %`) and falls off monotonically with
height (`-1.16 %` at 1 m, `< 0.1 %` above 15 m). Two effects stack: `mixing_length_fvm`
evaluates $\ell = \kappa z$ at the arithmetic face height instead of the log-mean height
that would make the finite-difference gradient exact for a log profile; and the
node-grid `U` is reconstructed from cell centres by a plain average,
`0.5*(U[i]+U[i+1])`, which further underestimates a concave profile. `closure_1_model_U_fvm_opa`
has the same near-ground error (`-4.52 %` at 0.25 m) because it evaluates the mixing
length at the same face locations — the two solvers differ in how they handle the
top boundary, not in this near-ground truncation error. Neither is wrong: this is the
same log-mean-height trade-off already on record for `fvm_opa`, and it is inherent to
solving a curved profile on a grid coarse relative to $z_0$.
